In [ ]:
import pandas as pd
import plotly.express as px

In [9]:
df = pd.read_csv("standings_data.csv")
df = df[df['manager'] != '0']


In [3]:
years = sorted(df["season"].unique(), reverse=True)
#selected_year = st.selectbox("Select year", years)
selected_year = 2026
filtered = df[df["season"] == selected_year]

fig = px.line(
    filtered,
    x="week",
    y="standings_rank",
    line_group="manager",
    color='manager',
    color_discrete_sequence=px.colors.qualitative.G10,
    labels={"week": "Week", 'standings_rank': 'standing'},
    title=f"Standings by Week — {selected_year}"
)
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(type='category')
fig

In [13]:
import numpy as np

In [62]:
tmp = df.copy()
tmp['streak_type'] = tmp['streak'].str[0]
tmp2 = tmp[['season', 'week', 'manager', 'streak_type']].copy()
tmp2['week'] = tmp2['week'] - 1
tmp2.rename(columns={'streak_type': 'next_streak_type'}, inplace=True)
tmp = tmp.merge(tmp2, on=['season', 'week', 'manager'], how='left')
tmp['max_streak'] = np.where(tmp['streak_type'] != tmp['next_streak_type'], tmp['streak'], None)
tmp['max_streak'] = np.where(tmp['max_streak'].isin(['L1', 'W1']), None, tmp['max_streak'])
tmp = tmp[tmp['max_streak'].notna()]
tmp['streak_length'] = tmp['max_streak'].str[1:].astype(int)
tmp

,season,week,manager,wins,losses,points_for,points_against,standings_rank,leader_wins,games_back,playoff_spot,streak,streak_type,next_streak_type,max_streak,streak_length
11,2021,2,Chris,2.0,0.0,965.35,806.83,2,2.0,0.0,1,W2,W,L,W2,2
18,2021,2,Garth,0.0,2.0,852.58,894.00,9,2.0,2.0,0,L2,L,W,L2,2
19,2021,2,Andy,0.0,2.0,727.33,957.50,10,2.0,2.0,0,L2,L,W,L2,2
22,2021,3,Rich,2.0,1.0,1337.76,1295.43,3,3.0,1.0,1,W2,W,L,W2,2
24,2021,3,Dave,1.0,2.0,1333.49,1332.58,5,3.0,2.0,0,L2,L,W,L2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1121,2026,4,Boyce,3.0,1.0,1495.33,1353.98,2,3.0,0.0,1,W3,W,L,W3,3
1131,2026,5,Andy,4.0,1.0,1799.15,1753.31,2,4.0,0.0,1,W2,W,NaN,W2,2
1132,2026,5,Sam,3.0,2.0,1859.33,1832.35,3,4.0,1.0,1,W2,W,NaN,W2,2
1138,2026,5,Chris,1.0,4.0,1621.68,1820.42,9,4.0,3.0,0,L3,L,NaN,L3,3


In [52]:
tmp.head(2)

,max_streak,number
2,L2,71
10,W2,60


In [58]:
streak_counts.head(2)

,streak_type,max_streak,number
0,L,L10,1
1,L,L11,1


In [ ]:
tmp.head(1)

In [64]:
streak_counts = tmp.groupby(['streak_type', 'streak_length']).size().reset_index().rename(columns={0: 'number'})
streak_counts.sort_values('number', inplace=True)
fig = px.funnel(streak_counts, x='streak_length', y='number', color='streak_type',
                    color_discrete_sequence=px.colors.qualitative.G10,
                    )
fig.show()

## make the loss labels more visible
## have a table below this for all streaks >= 6

In [79]:
def manager_cmap(df):
    custom_palette = [
        "#1b1f3b", "#283655", "#4d648d", "#1e434c", "#2e4600", "#486b00", "#7d4427",
        "#5e2129", "#2a3132", "#336666", "#003b46", "#07575b", "#66a5ad", "#2e003e", "#3a3153",
        "#1e1e20", "#22223b", "#3a506b", "#0b132b", "#1c2541", "#274690", "#576574", "#222f3e"
    ]
    if 'manager' in df.columns:
        df['manager_name'] = df['manager']
    if 'manager_name' not in df.columns:
        raise ValueError("DataFrame must contain 'manager_name' column for color mapping.")
    managers = df['manager_name'].unique()
    manager_color_map = {mgr: custom_palette[i % len(custom_palette)] for i, mgr in enumerate(managers)}
    return manager_color_map

def color_manager_row(row, cmap):
    manager = row['manager_name']
    color = cmap.get(manager, "#222222")
    return [f'background-color: {color}; color: white'] * len(row)

In [ ]:
streak_table = (tmp[tmp['streak_length'] >= 6]
                [['season', 'manager', 'streak_type', 'streak_length']]
                .sort_values(by=['streak_length', 'streak_type'], ascending=[False, False])
                )
streak_table_styled = streak_table.style.background_gradient(axis=0, gmap=streak_table["season"], cmap="Reds")
st.dataframe(streak_table_styled)

,season,manager,streak_type,streak_length
809,2024,Dave,L,11
1029,2025,Andy,L,11
429,2022,Rich,L,10
970,2025,Chris,W,9
309,2022,Sam,L,9
390,2022,Boyce,W,8
219,2021,Rich,L,8
609,2023,Andy,L,8
551,2023,Dave,W,7
417,2022,Garth,L,7


In [22]:

st.table(confusion_matrix)

In [ ]:
# standings over the season plot
# weeks in first
# plot of winning/losing streaks

In [ ]:
## hist H2H
test = matchup_week.copy()
test['teams'] = test.apply(lambda row: frozenset([row['manager_a'], row['manager_b']]), axis=1)
pd.pivot_table(test, index='teams', columns='winner', values='manager_a', aggfunc='count')

In [ ]:
e = team_week_points.groupby(['manager', 'season'])['points_week_rank'].mean().sort_values().reset_index()
pd.pivot_table(e, index='manager', columns='season', values='points_week_rank')

In [63]:
import plotly.express as px
import plotly.graph_objects as go

SEASON = 2025
WEEK = 2
tmp = matchup_week.copy()
highlight_mask = (tmp['season'] == SEASON) & (tmp['week'] == WEEK)
tmp['_dummy_x'] = 0

fig = px.strip(tmp,
               x='_dummy_x',
               y='margin_pct',
               color_discrete_sequence=['gray'],
               hover_data={'season': True, 'week': True, 'margin_pct': True, '_dummy_x': False},  # Only show what you want
            )
fig.data[0].hovertemplate = 'Season: %{customdata[0]}<br>Week: %{customdata[1]}<br>Margin %: %{y:.3f}<extra></extra>'
fig.data[0].customdata = tmp[['season', 'week']].values

# Overlay highlights in red
fig.add_scatter(
    x=tmp.loc[highlight_mask, '_dummy_x'],
    y=tmp.loc[highlight_mask, 'margin_pct'],
    mode='markers',
    marker=dict(color='red', size=12, symbol='star'),
    hovertemplate='Season: %{customdata[0]}<br>Week: %{customdata[1]}<br>Margin %: %{y:.3f}<extra></extra>',
    customdata=tmp.loc[highlight_mask, ['season', 'week']].values,

)
fig.update_xaxes(range=[-0.2, 0.2], showticklabels=False, title_text='')
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
hist_top20 = (player_top
    .query("rank <= 20")
    .groupby(['player_type', 'manager_name', 'season', 'week'])
    .size().reset_index(name='count')
)
for player_type in hist_top20['player_type'].unique():
    print(f"Top 20 {player_type}s by manager:")
    plt.hist(hist_top20[hist_top20['player_type'] == player_type]['count'], bins=range(0, 21), align='left')
    plt.xlim(0, 10)
    plt.show()

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

SEASON = 2025
WEEK = 2
tmp = hit.copy()
highlight_mask = (tmp['season'] == SEASON) & (tmp['week'] == WEEK)
tmp['_dummy_x'] = 0

for col in ['R', 'RBI', 'R', 'SB', 'BB', 'HBP', '1B', '2B', '3B', 'HR', 'TB']:
    fig = px.strip(tmp,
                x='_dummy_x',
                y=col,
                color_discrete_sequence=['gray'],
                hover_data={'season': True, 'week': True, col: True, '_dummy_x': False},  # Only show what you want
                )
    fig.data[0].hovertemplate = 'Season: %{customdata[0]}<br>Week: %{customdata[1]}<br>' + col + ': %{y:.3f}<extra></extra>'
    fig.data[0].customdata = tmp[['season', 'week']].values

    # Overlay highlights in red
    fig.add_scatter(
        x=tmp.loc[highlight_mask, '_dummy_x'],
        y=tmp.loc[highlight_mask, col],
        mode='markers',
        marker=dict(color='red', size=12, symbol='star'),
        hovertemplate='Season: %{customdata[0]}<br>Week: %{customdata[1]}<br>' + col + ': %{y:.3f}<extra></extra>',
        customdata=tmp.loc[highlight_mask, ['season', 'week']].values,

    )
    fig.update_xaxes(range=[-0.2, 0.2], showticklabels=False, title_text='')
    fig.update_layout(showlegend=False)
    fig.show()